# 02 - Files and GitHub (R)

**File:** `notebooks/02_files_and_github_r.ipynb`

**What this does:** Shows how a notebook finds folders on disk, writes a file, reads it back, and how that work gets to GitHub.

**How to run it:** Open this file in JupyterLab, check that the kernel shown in the
top-right corner says **R**, then choose *Run > Run All Cells*.

**Inputs:** none -- this notebook creates the file it reads

**Outputs:** `outputs/stations_r.csv` and `outputs/warm_stations_r.csv`

## Where am I? Finding the repo folder

A notebook runs from the folder it lives in (`notebooks/`), **not** from the top
of the repository. That trips people up constantly: a path that works in a
terminal at the repo root will not work here.

Rather than writing `../` everywhere, the cell below works out where the top of
the repo is once, and builds paths from there. Every notebook here uses this
same short block.

In [ ]:
# getwd() is the folder this notebook runs in.
repo <- getwd()
if (basename(repo) == "notebooks") repo <- dirname(repo)

cat("Repo folder:", repo, "\n")

## 1. What files are around me?

`list.files()` lists the contents of a folder. This is the notebook equivalent of
typing `ls` in a terminal.

In [ ]:
for (name in sort(list.files(repo))) {
  kind <- if (dir.exists(file.path(repo, name))) "folder" else "file"
  cat(sprintf("%-7s %s\n", kind, name))
}

## 2. Writing a file

Results go in an `outputs/` folder. Keeping outputs separate from inputs means a
mistake never destroys your original data -- a habit worth forming early.

Here we build a small table and save it as a CSV.

In [ ]:
stations <- data.frame(
  station_id   = c("ST-001", "ST-002", "ST-003", "ST-004", "ST-005"),
  name         = c("Kaena Point", "Penguin Bank", "Makapuu",
                   "Waianae Deep", "Kaiwi Channel"),
  latitude     = c(21.5760, 21.0400, 21.3100, 21.4200, 21.2600),
  longitude    = c(-158.2800, -157.3900, -157.6500, -158.3600, -157.7300),
  water_temp_c = c(25.4, 26.1, 24.8, 25.9, 26.4)
)

outputs <- file.path(repo, "outputs")
dir.create(outputs, showWarnings = FALSE)   # makes the folder, quiet if it exists

csv_path <- file.path(outputs, "stations_r.csv")
write.csv(stations, csv_path, row.names = FALSE)   # drop the row numbers

cat("Wrote", csv_path, "\n")
cat(file.info(csv_path)$size, "bytes\n")

## 3. Reading it back

`read.csv()` turns the file back into a data frame. This is the same call you
would use on any CSV, wherever it came from.

In [ ]:
loaded <- read.csv(csv_path)

cat(nrow(loaded), "rows,", ncol(loaded), "columns\n")
head(loaded)

## 4. Doing something with it

Keep only the warmer stations. Nothing here changes the file on disk -- `loaded`
is a copy held in memory.

In [ ]:
warm <- loaded[loaded$water_temp_c > 25.5, ]

cat(nrow(warm), "of", nrow(loaded), "stations are warmer than 25.5 C\n")
warm[, c("station_id", "name", "water_temp_c")]

## 5. Saving the result

Write it alongside the first file, under a different name.

In [ ]:
result_path <- file.path(outputs, "warm_stations_r.csv")
write.csv(warm, result_path, row.names = FALSE)

cat("Wrote", result_path, "\n")

## 6. Getting your work back to GitHub

`system()` runs a shell command, so you can check on git without leaving Jupyter:

In [ ]:
cat(system("git status --short", intern = TRUE), sep = "\n")

Notice what is **not** there: the files you just wrote to `outputs/`. That folder
is listed in [`.gitignore`](../.gitignore), so git ignores it on purpose --
results are something you can always regenerate by re-running the notebook, and
data files are usually too big for a repository anyway.

What you probably *do* see is this notebook itself marked `M` for modified. You
only ran it -- but running a notebook stores its output inside the file, so the
file really did change.

To save your work back to GitHub, run these in a terminal
(*File > New > Terminal* in JupyterLab):

```bash
git checkout -b your-name-your-feature    # once, before you start
git add .
git commit -m "a short note about what you did"
git push -u origin your-name-your-feature
```

Everyone works on their own branch here -- no forking. The full walkthrough is in
the [main README](../README.md).

Two things worth knowing:

- **Notebooks produce noisy diffs.** A notebook stores its outputs inside the
  file, so re-running it shows up as a change even when you edited nothing.
  Running *Kernel > Restart Kernel and Clear Outputs* before committing keeps
  pull requests readable.
- **Keep data files out of git.** They are usually too big, and GitHub rejects
  anything over 100 MB. See [`.gitignore`](../.gitignore).

## Done

Next: **`03_aquaview_stac_r.ipynb`**, which pulls real data off the internet.